# Offline Intraday Realized Variance Walkthrough

This notebook answers one question: can simple lagged realized-variance features forecast next-day realized variance better than naive baselines?

Everything in this notebook is offline-first. The runtime path reads only tracked parquet files under `data/raw/`. The optional ClickHouse extraction step is a one-time cache population path and is not called here.


In [1]:
from pathlib import Path

from intraday_pnl_explain.data_access.raw_manifest import load_raw_manifest

candidate_root = Path.cwd().resolve()
if candidate_root.name == "notebooks":
    candidate_root = candidate_root.parent
project_root = candidate_root

raw_root = project_root / "data" / "raw"
manifest = load_raw_manifest(raw_root=raw_root)

manifest


RawManifest(dataset_name='intraday_variance_demo', symbols=['AAPL', 'MSFT', 'NVDA', 'XOM', 'CVX', 'JPM'], start_date='2026-03-24', end_date='2026-04-06', bar_frequency='1min', timezone='UTC', trading_session_timezone='America/New_York', trading_session='09:30-15:59', row_count=23400, file_count=6, total_bytes=416108, required_columns=['symbol', 'timestamp', 'price', 'volume'], partition_layout='data/raw/intraday_bars/symbol=<SYMBOL>/part-000.parquet')

## Raw Parquet Contract

The tracked manifest defines the symbol universe, date coverage, bar frequency, timezone, and schema.

Required bar columns are:

- `symbol`: ticker identifier.
- `timestamp`: intraday bar timestamp in Coordinated Universal Time (UTC).
- `price`: intraday price used for return construction.
- `volume`: optional liquidity proxy used for diagnostics.


In [2]:
import pandas as pd

manifest_summary = pd.DataFrame(
    [
        {
            "dataset_name": manifest.dataset_name,
            "symbol_count": len(manifest.symbols),
            "start_date": manifest.start_date,
            "end_date": manifest.end_date,
            "bar_frequency": manifest.bar_frequency,
            "timezone": manifest.timezone,
            "row_count": manifest.row_count,
            "file_count": manifest.file_count,
            "total_bytes": manifest.total_bytes,
        }
    ]
)
manifest_summary


,dataset_name,symbol_count,start_date,end_date,bar_frequency,timezone,row_count,file_count,total_bytes
0,intraday_variance_demo,6,2026-03-24,2026-04-06,1min,UTC,23400,6,416108


## Intraday Return and Realized Variance Theory

For intraday bar time index $t$, let $P_t$ be the bar price and define log return as $r_t = \log(P_t / P_{t-1})$.

For one symbol and one trading day $d$, realized variance is the sum of squared intraday log returns:

$$RV_d = \sum_{t \in d} r_t^2$$

This target is non-negative by construction and increases when intraday move magnitudes increase.


In [3]:
from intraday_pnl_explain.data_access.raw_bars import load_raw_intraday_bars
from intraday_pnl_explain.realized_variance.construct import construct_daily_realized_variance

raw_bars = load_raw_intraday_bars(raw_root=raw_root, manifest=manifest)
rv_daily = construct_daily_realized_variance(bars=raw_bars)

rv_daily.head()


,symbol,date,realized_variance,bar_count,realized_volatility,is_complete_session
0,AAPL,2026-03-24,1.504150e-06,389,0.001226,True
1,AAPL,2026-03-25,1.022484e-06,389,0.001011,True
2,AAPL,2026-03-26,3.038023e-06,389,0.001743,True
3,AAPL,2026-03-27,3.438199e-07,389,0.000586,True
4,AAPL,2026-03-30,4.477872e-06,389,0.002116,True


## Feature Design and Leakage Control

We model next-day log realized variance. Define $y_{d+1} = \log(RV_{d+1})$.

Features on day $d$ include:

- lag-1 log realized variance,
- 5-day rolling mean of log realized variance,
- 5-day rolling standard deviation of log realized variance,
- previous-day range proxy from absolute log-RV change,
- bar completeness ratio.

No feature uses $RV_{d+1}$ or later information.


In [4]:
from intraday_pnl_explain.features.build_features import build_feature_table

feature_table = build_feature_table(rv_daily=rv_daily)
feature_table.head()


,symbol,feature_date,target_date,lag_1_log_rv,lag_5_mean_log_rv,lag_5_std_log_rv,prev_day_range_proxy,bar_completeness,target_log_rv_next_day
0,AAPL,2026-03-30,2026-03-31,-12.316363,-13.420874,1.001315,2.566785,1.0,-16.169420
1,AAPL,2026-03-31,2026-04-01,-16.169420,-13.973302,1.584217,3.853058,1.0,-13.132800
2,AAPL,2026-04-01,2026-04-02,-13.132800,-13.841207,1.629859,3.036621,1.0,-14.100852
3,AAPL,2026-04-02,2026-04-03,-14.100852,-14.120516,1.500879,0.968052,1.0,-14.368711
4,AAPL,2026-04-03,2026-04-06,-14.368711,-14.017629,1.452379,0.267859,1.0,-16.133568


## Walk-Forward Training and Baselines

Walk-forward evaluation means train on earlier dates and test on the next unseen date.

Model set:

- Persistence baseline: predict tomorrow with today.
- Rolling-mean baseline: predict tomorrow with recent average.
- Ridge regression: linear model on standardized features.

This setup is intentionally simple and interview-defensible.


In [5]:
from intraday_pnl_explain.evaluation.metrics import compute_model_metrics
from intraday_pnl_explain.modeling.train import build_walk_forward_predictions

predictions, coefficients = build_walk_forward_predictions(
    feature_table=feature_table,
    min_train_dates=4,
    ridge_alpha=1.0,
)
metrics = compute_model_metrics(predictions=predictions)

metrics


{'persistence': {'rmse': 1.299903109325107,
  'mae': 1.2091040059559646,
  'r2_oos': -1.2527716837695722},
 'ridge': {'rmse': 1.1900468648383082,
  'mae': 1.0886016326530006,
  'r2_oos': -0.8880928279861924},
 'rolling_mean': {'rmse': 1.6037040493151975,
  'mae': 1.426658702552076,
  'r2_oos': -2.428811932374248}}

## End-to-End Offline Pipeline Artifacts

The project ships one reproducible command path that writes non-HTML artifacts:

- `metrics.json` for evaluation summary,
- `predictions.parquet` for model outputs,
- `coefficients.csv` for linear model interpretation,
- `figures/*.png` for static diagnostics.


In [6]:
from intraday_pnl_explain.pipeline.run_offline_demo import run_offline_demo

demo_output_directory = project_root / "outputs" / "demo_run"
run_offline_demo(output_directory=demo_output_directory)

sorted(path.relative_to(demo_output_directory).as_posix() for path in demo_output_directory.rglob("*") if path.is_file())


['coefficients.csv',
 'figures/coefficient_bar_chart.png',
 'figures/prediction_vs_actual.png',
 'figures/realized_variance_history.png',
 'figures/residual_distribution.png',
 'metrics.json',
 'predictions.parquet']